# Анализ отзывов «Золотого Яблока»

Исследование пользовательских отзывов: разведочный анализ, очистка текстов, определение языка, анализ тональности и подготовка признаков для выделения лояльных и нелояльных клиентов.

In [ ]:
!pip install gdown
!pip install openpyxl
# !pip install langdetect

In [ ]:
import pandas as pd

sheet_id = "1YRFWn3NuCioexFGyYKlCzE3uvfdOWHUl"
sheet_name = "data"

url = (
    f"https://docs.google.com/spreadsheets/d/{sheet_id}/export"
    f"?format=xlsx"
)

df = pd.read_excel(
    url,
    sheet_name=sheet_name,
    engine="openpyxl"
)

df.head()


,Pros,Cons,Comment,IsRecommended,Stars,CatalogName,ProductType,CreatedDate
0,приятно пахнет,после нанесения начала жечь лицо. кожу лица ст...,NaN,0,2,Power Nap SUPER MOISTURE EMBO HYDROGEL MASK,маски для лица,04/01/2025 00:25:15
1,супер очищение,нет,очень хорошо очищает от загрязнений и черных т...,1,5,Power Nap SUPER MOISTURE EMBO HYDROGEL MASK,маски для лица,04/01/2025 00:46:25
2,"самая классная маска, беру несколько раз от да...",NaN,буду заказывать еще!,1,5,Pury Fly SELF-BUBBLING COCONUT CHARCOAL MASK,маски для лица,04/01/2025 00:59:01
3,"Прекрасная маска, очень люблю ее использовать ...",NaN,NaN,1,5,Magic Glow REVITALIZING MASK,маски для лица,04/01/2025 01:05:30
4,Самые лучшие патчи которыми я когда-то пользов...,нет,NaN,1,5,LIFT ME UP,патчи для глаз,04/01/2025 01:09:16


In [ ]:
file_id = "1spzzyEutsZc7Bo2ZqANd_S-97UJY-vil"

url = f"https://drive.google.com/uc?export=download&id={file_id}"

df_with_sentiment = pd.read_csv(url) # при необходимости: encoding="utf-8", sep=";", etc.
df_with_sentiment.head()


,Pros,Cons,Comment,IsRecommended,Stars,CatalogName,ProductType,CreatedDate,cons_norm,cons_is_empty,Cons_clean,pros_norm,pros_is_empty,Pros_clean,pros_sentiment,cons_sentiment,comment_sentiment,Sentiment
0,приятно пахнет,после нанесения начала жечь лицо. кожу лица ст...,NaN,0,2,Power Nap SUPER MOISTURE EMBO HYDROGEL MASK,маски для лица,04/01/2025 00:25:15,после нанесения начала жечь лицо кожу лица стя...,False,после нанесения начала жечь лицо. кожу лица ст...,приятно пахнет,False,приятно пахнет,Positive,Negative,NaN,0
1,супер очищение,нет,очень хорошо очищает от загрязнений и черных т...,1,5,Power Nap SUPER MOISTURE EMBO HYDROGEL MASK,маски для лица,04/01/2025 00:46:25,нет,True,NaN,супер очищение,False,супер очищение,Neutral,Negative,Positive,1
2,"самая классная маска, беру несколько раз от да...",NaN,буду заказывать еще!,1,5,Pury Fly SELF-BUBBLING COCONUT CHARCOAL MASK,маски для лица,04/01/2025 00:59:01,NaN,True,NaN,самая классная маска беру несколько раз от дар...,False,"самая классная маска, беру несколько раз от да...",Positive,NaN,Positive,1
3,"Прекрасная маска, очень люблю ее использовать ...",NaN,NaN,1,5,Magic Glow REVITALIZING MASK,маски для лица,04/01/2025 01:05:30,NaN,True,NaN,прекрасная маска очень люблю ее использовать п...,False,"Прекрасная маска, очень люблю ее использовать ...",Positive,NaN,NaN,1
4,Самые лучшие патчи которыми я когда-то пользов...,нет,NaN,1,5,LIFT ME UP,патчи для глаз,04/01/2025 01:09:16,нет,True,NaN,самые лучшие патчи которыми я когда то пользов...,False,Самые лучшие патчи которыми я когда-то пользов...,Positive,Negative,NaN,1


## 1. Разведочный анализ данных

### Базовые статистики

In [ ]:
# Преобразуем CreatedDate в datetime
df['CreatedDate'] = pd.to_datetime(df['CreatedDate'], format='%m/%d/%Y %H:%M:%S')

# Создаём период месяца (например, '2025-04')
df['ReviewMonth'] = df['CreatedDate'].dt.to_period('M')

monthly_stats = df.groupby(['CatalogName', 'ReviewMonth']).agg(
    avg_stars=('Stars', 'mean'),
    std_stars=('Stars', 'std'),      # стандартное отклонение — чем меньше, тем стабильнее
    review_count=('Stars', 'count')  # сколько отзывов за месяц
).reset_index()


product_trends = monthly_stats.groupby('CatalogName').agg(
    overall_avg_stars=('avg_stars', 'mean'),
    avg_std_stars=('std_stars', 'mean'),  # среднее отклонение по месяцам
    min_monthly_stars=('avg_stars', 'min'), # самый плохой месяц
    max_monthly_stars=('avg_stars', 'max'), # лучший месяц
    months_with_reviews=('ReviewMonth', 'nunique'), # сколько месяцев были отзывы
    total_reviews=('review_count', 'sum')   # общее количество отзывов
).reset_index()

# Чем ниже std и выше min — тем стабильнее
product_trends['stability_score'] = (
    (5 - product_trends['avg_std_stars']) / 5  # нормализуем std (чем ниже — тем лучше)
    + (product_trends['min_monthly_stars'] / 5)  # мин. рейтинг должен быть высоким
) / 2  # среднее двух компонентов

product_trends

,CatalogName,overall_avg_stars,avg_std_stars,min_monthly_stars,max_monthly_stars,months_with_reviews,total_reviews,stability_score
0,3-D Cure BIO CELLULOSE MASK FOR RESTORATIVE SK...,4.390873,1.214893,4.050000,4.785714,7,127,0.783511
1,Anti-Gravity,4.487879,1.077594,4.111111,4.777778,7,442,0.803352
2,Aurora Rich,4.692060,0.707064,4.551724,4.883333,7,186,0.884466
3,BROW FICTION,3.691511,1.512313,3.413333,4.296296,7,473,0.690102
4,BROW GURU,4.696636,0.768917,4.465517,4.853211,7,770,0.869660
...,...,...,...,...,...,...,...,...
168,Wet Kiss,4.742941,0.668144,4.724138,4.761745,2,820,0.905599
169,Your skin's best friends,4.860763,0.372447,4.666667,5.000000,6,59,0.929422
170,everyday hero,4.290797,1.027390,3.400000,4.714286,7,76,0.737261
171,lash cocoon,4.503830,1.014604,4.380645,4.813403,7,5548,0.836604


In [ ]:
import pandas as pd
import numpy as np



# Объединяем по CatalogName, оставляя все исходные строки (left join)
df = df.merge(df_with_sentiment[['CatalogName', 'Sentiment']], on='CatalogName', how='left')


# --- 2. Вспомогательная функция ---
def is_empty_or_no(series):
    """
    Возвращает True, если строка пропущена, пустая или содержит только 'нет', 'Нет' и т.п.
    """
    series = series.fillna('').astype(str).str.strip().str.lower()
    return (series == '') | (series == 'нет') | (series == 'no') | (series == '-')

# --- 3. Преобразование даты и признаков ---
df['CreatedDate'] = pd.to_datetime(df['CreatedDate'], format='%m/%d/%Y %H:%M:%S')
df['ReviewMonth'] = df['CreatedDate'].dt.to_period('M')

df['is_positive_review'] = df['Stars'] >= 4
df['has_no_pros'] = is_empty_or_no(df['Pros'])
df['has_no_cons'] = is_empty_or_no(df['Cons'])

# --- 4. Сводка по каталогам ---
catalog_summary = (
    df.groupby('CatalogName')
    .agg(
        review_count=('Stars', 'count'),
        avg_recommendation_rate=('IsRecommended', 'mean'),
        avg_sentiment_rate=('Sentiment', 'mean'),  # <-- новая метрика
        avg_positive_stars_rate=('is_positive_review', 'mean'),
        avg_stars=('Stars', 'mean')
    )
    .reset_index()
)

# --- 5. Месячная статистика (для стабильности) ---
monthly_stats = df.groupby(['CatalogName', 'ReviewMonth']).agg(
    avg_stars=('Stars', 'mean'),
    std_stars=('Stars', 'std')
).reset_index()

# --- 6. Агрегация по стабильности ---
time_stability = monthly_stats.groupby('CatalogName').agg(
    min_monthly_stars=('avg_stars', 'min'),
    max_monthly_stars=('avg_stars', 'max'),
    avg_std_stars=('std_stars', 'mean'),
    months_with_reviews=('ReviewMonth', 'nunique')
).reset_index()

# Заполняем NaN в avg_std_stars (std для одного отзыва = NaN → заменяем на 0)
time_stability['avg_std_stars'] = time_stability['avg_std_stars'].fillna(0)

# Рассчитываем stability_score
time_stability['stability_score'] = (
    (5 - time_stability['avg_std_stars']) / 5 +
    time_stability['min_monthly_stars'] / 5
) / 2

# Ограничиваем в [0, 1]
time_stability['stability_score'] = time_stability['stability_score'].clip(0, 1)

# --- 7. Объединяем итоговые данные ---
final_report = catalog_summary.merge(
    time_stability[['CatalogName', 'stability_score']],
    on='CatalogName',
    how='left'
)

# Заполняем отсутствующую стабильность (если мало данных)
final_report['stability_score'] = final_report['stability_score'].fillna(0)

# --- 8. Выбор и сортировка колонок ---
final_report = final_report[[
    'CatalogName',
    'review_count',
    'avg_recommendation_rate',
    'avg_sentiment_rate',          # <-- добавлено
    'avg_stars',
    'avg_positive_stars_rate',
    'stability_score'
]]

# Сортировка: сначала по количеству отзывов (убывание), потом по рекомендациям (убывание)
final_report = final_report.sort_values(
    by=['review_count', 'avg_recommendation_rate'],
    ascending=[False, False]
).reset_index(drop=True)

# --- Вывод ---
final_report

### Определение языков отзывов

---



In [ ]:
# допустимые алфавиты
def is_ru_en_letter(ch: str) -> bool:
    return (
        'a' <= ch <= 'z' or
        'A' <= ch <= 'Z' or
        'а' <= ch <= 'я' or
        'А' <= ch <= 'Я' or
        ch in ('ё', 'Ё')
    )


def has_non_ru_en_letters(text) -> bool:
    if not isinstance(text, str):
        return False

    for ch in text:
        if ch.isalpha() and not is_ru_en_letter(ch):
            return True
    return False


cols = ['Pros', 'Cons', 'Comment']

mask = df[cols].apply(
    lambda row: any(has_non_ru_en_letters(cell) for cell in row),
    axis=1
)

df_filtered = df[mask]

df_filtered[['Pros', 'Cons', 'Comment']]

,Pros,Cons,Comment
5480,"красивый цвет, приятный запах, есть ещë 02 отт...",--,приятный холодок на губах)
8659,Керемет ұзартады,Жоқ,NaN
8771,"жақсы термотушь, көз астында бояу қалдырмайды....",дарлингтан соң басқа тушьтерге қарай алмаймын,NaN
10198,ағып кетпи жақсы тұрады,но жабысып калады и ауырсынады,NaN
10778,ағып кетпейді,жоқ,маған ұнайды
...,...,...,...
89974,Стойкость.,Их нет,Наверное это единственная достойная тушь. Брал...
92434,"прекрасно удлиняет ресницы, не склеивает, дает...",NaN,NaN
94616,иісі өте жағымды қатты ұнады😍,NaN,NaN
94642,"Очень классная тушь, очень еë люблю за то что ...",Нет,NaN


русский, английский, казахский и смешанные тексты на этих языках

In [ ]:
import numpy as np

cols = ['Pros', 'Cons', 'Comment']

# 1) собираем весь текст строки в одно поле
text = (
    df[cols]
    .fillna('')
    .astype(str)
    .agg(' '.join, axis=1)
)

# 2) детектим "присутствие" алфавитов
has_latin = text.str.contains(r'[A-Za-z]', regex=True)

# базовая кириллица (включая Ё/ё)
has_cyr = text.str.contains(r'[А-Яа-яЁё]', regex=True)

# "казахские" специфические кириллические буквы
has_kk_extra = text.str.contains(r'[ӘәҒғҚқҢңӨөҰұҮүҺһІі]', regex=True)

# маркеры "русскости" (чтобы отличить kk-only от ru+kk внутри кириллицы — грубо, но полезно)
has_ru_markers = text.str.contains(r'[ЁёЪъЩщЭэ]', regex=True)

# 3) категория
lang = np.select(
    [
        has_latin & ~has_cyr,                                  # только английский (латиница)
        has_cyr & ~has_latin & ~has_kk_extra,                  # только русский (кириллица без kk-спецбукв)
        has_cyr & ~has_latin & has_kk_extra & ~has_ru_markers, # только казахский (кириллица с kk-спецбуквами)
        (has_cyr & has_latin) | (has_kk_extra & has_ru_markers)# смесь (ru+en, kk+en, ru+kk)
    ],
    ['en', 'ru', 'kk', 'mixed'],
    default='other/empty'  # нет букв или что-то нестандартное
)

df['lang_cat'] = lang

# 4) сколько строк в каждой категории
counts = df['lang_cat'].value_counts()
print(counts)


lang_cat
ru             93184
mixed           1491
kk                59
other/empty        9
en                 3
Name: count, dtype: int64


### Очищение минусов и плюсов от "нет"


In [ ]:
# Полный пайплайн очистки текстовых полей
!pip install rapidfuzz
import re
import unicodedata
import pandas as pd
from rapidfuzz import fuzz

# -------------------------
# 1) Normalization
# -------------------------
REPEAT_RE = re.compile(r"(.)\1{2,}", flags=re.UNICODE)
CLEAN_RE = re.compile(r"[^0-9a-zа-яёіїєґқңөұүһәә\s]+", flags=re.IGNORECASE | re.UNICODE)

def squash_repeats(s: str, keep: int = 1) -> str:
    return REPEAT_RE.sub(lambda m: m.group(1) * keep, s)

def normalize_text(x) -> str:
    if pd.isna(x):
        return ""
    s = unicodedata.normalize("NFKC", str(x))
    s = s.replace("ё", "е").lower().strip()
    s = CLEAN_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = squash_repeats(s, keep=1)
    s = re.sub(r"\s+", " ", s).strip()

    # --- corpus-specific micro-normalizations ---
    # "нетуу", "неееет" already handled by repeats; but keep some aliases:
    # normalize common variants/typos
    s = re.sub(r"\bобьем\b", "объем", s)   # "обьем" -> "объем"
    s = re.sub(r"\bнету\b", "нету", s)     # no-op, just explicit

    # normalize weird constructions seen in your top:
    s = re.sub(r"^(из)\s+(нету|нет)$", r"\2", s)  # "из нет" -> "нет"
    s = re.sub(r"^(их)\s+не$", "нет", s)          # "их не" -> "нет"
    return s

# -------------------------
# 2) Список пустых по смыслу формулировок из текущего корпуса
# -------------------------
EMPTY_EXACT = {
    # strict empties / placeholders
    "", "-", "—", "–", "_", "0", "...", "…", "--",

    # RU core
    "нет", "нету", "не",
    "нету вообще",
    "вроде нет",
    "пока нет",
    "никаких",
    "нет их", "их нет", "их просто нет", "для меня их нет",
    "не найдено", "не найдены", "не нашлось",
    "не замечено", "не выявлены", "не имеется", "не имеет",
    "не вижу", "супер",  # (супер часто как "всё отлично, минусов нет" — если это не ок, убери)

    # "я не нашла" как самостоятельная фраза
    "я не нашла",
}

# Multilang standalone "no"
NO_WORDS = {
    "no", "none", "nothing", "na", "n a", "n/a",
    "жок", "жоқ",
    "нема", "немає", "няма", "ні",
    "nie", "nein", "non"
}

# -------------------------
# 3) Patterns: empty-by-meaning
# -------------------------
# If "нет X" and X is not about минусы/недостатки, it is a real con: "нет эффекта"
NEGATION_WITH_OBJECT = re.compile(r"^нет\s+\S+", flags=re.IGNORECASE)

# If it explicitly says "нет минусов/недостатков" -> empty
ALLOW_NO_CONS_OBJECT = re.compile(r"^нет\s+(минус|минусов|недостаток|недостатков)\b", flags=re.IGNORECASE)

EMPTY_PATTERNS = re.compile(
    r"^(нет|нету|не)$|"
    r"^(минус(ов)?|недостат(ок|ков)) нет$|"
    r"^нет (минус(ов)?|недостат(ок|ков))$|"
    r"^без (минус(ов)?|недостат(ок|ков))$|"
    r"^их нет$|"
    r"^их просто нет$|"
    r"^не (наш(ел|ла|ли)|нашлось|найдено|найдены|замет(ил|ила|или)|обнаруж(ил|ила|или)|обнаружено|выяв(ил|ила|или)|выявлено)$|"
    r"^пока не (нашл(а|и)?|заметил(а|и)?|обнаружил(а|и)?|обнаружено|выявил(а|и)?|выявлено|увидел(а|и)?)$|"
    r"^недостатков не (нашл(а|и)?|заметил(а|и)?|обнаружил(а|и)?|обнаружено)$|"
    r"^отсутств(ует|уют)$|"
    r"^отсутств(ует|уют)\s+(минус(ы|ов)|недостат(ок|ки|ков))$|"
    r"^никаких (минус(ов)?|недостат(ок|ков)) нет$|"
    r"^все (хорошо|нормально|ок)$|"
    r"^нет все отлично$",
    flags=re.IGNORECASE
)

# Canonical phrases for fuzzy (only short strings)
CANONICAL = [
    "нет", "нету", "их нет", "их просто нет",
    "минусов нет", "недостатков нет", "без минусов", "без недостатков",
    "не обнаружено", "не обнаружил", "не обнаружила",
    "не нашел", "не нашла", "не найдено", "не найдены", "не нашлось",
    "не заметил", "не заметила", "не замечено",
    "не выявлены", "не выявлено",
    "пока не заметила", "пока не нашла", "пока не обнаружила",
    "вроде нет", "пока нет", "нету вообще",
    "не увидела","для меня нет","нету недостатков","нет таких","нет такого","нетуу","не нашла недостатков","не заметила недостатков",
    "нет все супер", 'нет таких','ничего','нету все отлично','таких нет', 'нет все супер', 'неа', 'неи','норм', 'ноу'
 'отсутсвуют', 'нету никаких','я их не нашла', 'нетт','неь','нет такого','нет никаких','отсутсвуют'
]

def fuzzy_empty(norm: str, threshold: int = 92) -> bool:
    if len(norm) > 45:
        return False
    best = max(fuzz.ratio(norm, c) for c in CANONICAL)
    return best >= threshold

# -------------------------
# 4) Final classifier
# -------------------------
def is_empty(x) -> bool:
    norm = normalize_text(x)

    # exact hardcoded (includes placeholders)
    if norm in EMPTY_EXACT:
        return True

    # multilang standalone no
    if norm in NO_WORDS:
        return True

    # exact patterns
    if EMPTY_PATTERNS.match(norm):
        return True

    # protect against "нет эффекта / нет объема / нет скидки" etc. => REAL con
    # but allow "нет минусов/недостатков"
    if NEGATION_WITH_OBJECT.match(norm) and not ALLOW_NO_CONS_OBJECT.match(norm):
        return False

    # fuzzy fallback for short variants / typos
    if fuzzy_empty(norm, threshold=92):
        return True

    return False

# -------------------------
# 5) Apply
# -------------------------
df["pros_norm"] = df["Pros"].map(normalize_text)
df["pros_is_empty"] = df["Pros"].map(is_empty)
df["Pros_clean"] = df["Pros"].where(~df["pros_is_empty"], pd.NA)

df["cons_norm"] = df["Cons"].map(normalize_text)
df["cons_is_empty"] = df["Cons"].map(is_empty)
df["Cons_clean"] = df["Cons"].where(~df["cons_is_empty"], pd.NA)

# -------------------------
# 6) Debug views
# -------------------------
print("Empty share:", round(df["cons_is_empty"].mean() * 100, 2), "%")

print("\nTop EMPTY (normalized):")
#print(df.loc[df["cons_is_empty"], "cons_norm"].value_counts().head(30))

print("\nTop SHORT NON-empty candidates to review (normalized):")
cand = df.loc[~df["cons_is_empty"], "cons_norm"]
display(cand[cand.str.len().between(1, 35)].value_counts().head(300).reset_index().cons_norm.tolist())


Empty share: 80.75 %

Top EMPTY (normalized):

Top SHORT NON-empty candidates to review (normalized):


['цена',
 'быстро заканчивается',
 'маленький объем',
 'быстро засыхает',
 'склеивает ресницы',
 'быстро сохнет',
 'быстрый расход',
 'быстро высыхает',
 'дорого',
 'дороговато',
 'осыпается',
 'стоимость',
 'липкий',
 'запах',
 'большой расход',
 'сушит губы',
 'сушит кожу',
 'мало продукта',
 'мало',
 'дорогая',
 'нет эффекта',
 'быстро кончается',
 'скатывается',
 'немного осыпается',
 'никакого эффекта',
 'немного сушит губы',
 'не увлажняет',
 'цена без скидки',
 'сползают',
 'немного липкий',
 'для себя не нашла',
 'упаковка',
 'быстро закончилась',
 'нет таких',
 'цена завышена',
 'не фиксирует',
 'цена кусается',
 'немного липкая',
 'нет такого',
 'сыпется',
 'очень быстро заканчивается',
 'сушит',
 'липкая',
 'быстро закончился',
 'быстро заканчиваются',
 'нет объема',
 'быстро расходуется',
 'очень маленький объем',
 'не стойкий',
 'неудобная упаковка',
 'плохо увлажняет',
 'сушит ресницы',
 'объем',
 'завышенная цена',
 'эффекта нет',
 'цвет',
 'иногда осыпается',
 'немного 

## 2. Анализ тональности

### Классификация тональности с помощью модели Hugging Face

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "tabularisai/multilingual-sentiment-analysis"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Берём названия классов из конфига (надежнее, чем хардкодить)
# Если вдруг там будет LABEL_0..LABEL_4, ниже есть fallback.
id2label = getattr(model.config, "id2label", None) or {}
fallback_map = {0: "Very Negative", 1: "Negative", 2: "Neutral", 3: "Positive", 4: "Very Positive"}

def _label_from_id(idx: int) -> str:
    lab = id2label.get(idx, str(idx))
    # если конфиг даёт "LABEL_0", "LABEL_1"... — заменим на человекочитаемое
    if isinstance(lab, str) and lab.upper().startswith("LABEL_"):
        return fallback_map.get(idx, lab)
    return lab if isinstance(lab, str) else str(lab)

@torch.no_grad()
def predict_sentiment_batch(texts, batch_size: int = 32, max_length: int = 512):
    """
    texts: list[str] (НЕ NaN)
    returns: list[str] labels
    """
    labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(device)

        logits = model(**enc).logits
        pred_ids = torch.argmax(logits, dim=-1).tolist()
        labels.extend([_label_from_id(j) for j in pred_ids])

    return labels

def add_sentiment_columns(df: pd.DataFrame,
                          cols=("Pros", "Cons", "Comment"),
                          batch_size: int = 32) -> pd.DataFrame:
    df_out = df.copy()

    for col in cols:
        s = df_out[col].fillna("").astype(str)
        mask = s.str.strip().ne("")  # только непустые

        sentiments = pd.Series([np.nan] * len(df_out), index=df_out.index, dtype="object")

        if mask.any():
            texts = s[mask].tolist()
            preds = predict_sentiment_batch(texts, batch_size=batch_size)
            sentiments.loc[mask] = preds

        new_col = f"{col.lower()}_sentiment"  # pros_sentiment, cons_sentiment, comment_sentiment
        df_out[new_col] = sentiments

    return df_out

# применение:
df_with_sentiment = add_sentiment_columns(df, cols=("Pros_clean", "Cons_clean", "Comment"), batch_size=64)
df_with_sentiment[["pros_clean_sentiment", "cons_clean_sentiment", "comment_sentiment"]].head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

,pros_clean_sentiment,cons_clean_sentiment,comment_sentiment
0,Positive,Negative,NaN
1,Neutral,NaN,Positive
2,Positive,NaN,Positive
3,Positive,NaN,NaN
4,Positive,NaN,NaN


### Лемматизация и подготовка текста

In [ ]:
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 97.6 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
import pymorphy3

# если NLTK-стопслова нужны
try:
    import nltk
    from nltk.corpus import stopwords
    try:
        _ = stopwords.words("russian")
    except Exception:
        nltk.download("stopwords", quiet=True)
except Exception:
    stopwords = None

# ---------- 0) оставляем нужные колонки ----------
keep_cols = [
    "Pros_clean", "Cons_clean", "Comment",
    "pros_clean_sentiment", "cons_clean_sentiment", "comment_sentiment",
    "IsRecommended", "Stars", "CatalogName", "ProductType", "CreatedDate"
]

df_out = df_with_sentiment[keep_cols].copy()

# ---------------- text utils (суть та же) ----------------

def to_lower(text: str) -> str:
    return text.lower()

def extract_emojis(text: str) -> tuple[str, str]:
    emoji_re = re.compile(
        "["

        "\U0001F300-\U0001F5FF"
        "\U0001F600-\U0001F64F"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002700-\U000027BF"
        "\U00002600-\U000026FF"
        "]+",
        flags=re.UNICODE,
    )
    emojis = emoji_re.findall(text)
    text_wo = emoji_re.sub(" ", text)
    return text_wo, " ".join(emojis).strip()

def replace_hyphen_with_space(text: str) -> str:
    return re.sub(r"[-‐-‒–—―]+", " ", text)

def remove_symbols_keep_ru_en(text: str) -> str:
    # оставляем кириллицу/латиницу/пробелы (цифры и прочее убираем)
    text = re.sub(r"[^A-Za-zА-Яа-яЁё\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_ru_stopwords() -> set:
    try:
        if stopwords is None:
            return set()
        return set(stopwords.words("russian"))
    except Exception:
        return set()

RU_SW = get_ru_stopwords()

# 2) инициализация лемматизатора (1 раз)
_RU_MORPH = pymorphy3.MorphAnalyzer()

def remove_stopwords_ru(text: str) -> str:
    if not RU_SW:
        return text
    return " ".join([t for t in text.split() if t not in RU_SW])

def lemmatize_ru(text: str) -> str:
    if _RU_MORPH is None:
        return text
    toks = text.split()
    return " ".join(_RU_MORPH.parse(t)[0].normal_form for t in toks)

def preprocess_cell_ru(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    text = str(x).strip()
    if not text:
        return ""

    text = to_lower(text)
    text, _ = extract_emojis(text)
    text = replace_hyphen_with_space(text)
    text = remove_symbols_keep_ru_en(text)

    text = remove_stopwords_ru(text)
    text = lemmatize_ru(text)

    return re.sub(r"\s+", " ", text).strip()

# ---------- 3) добавляем лемматизированные колонки ----------
df_out["Pros_clean_lemmatized"] = df_out["Pros_clean"].apply(preprocess_cell_ru)
df_out["Cons_clean_lemmatized"] = df_out["Cons_clean"].apply(preprocess_cell_ru)
df_out["Comment_lemmatized"]   = df_out["Comment"].apply(preprocess_cell_ru)

print("lemmatizer_loaded =", _RU_MORPH is not None)
df_out.head(5)


lemmatizer_loaded = True


,Pros_clean,Cons_clean,Comment,pros_clean_sentiment,cons_clean_sentiment,comment_sentiment,IsRecommended,Stars,CatalogName,ProductType,CreatedDate,Pros_clean_lemmatized,Cons_clean_lemmatized,Comment_lemmatized
0,приятно пахнет,после нанесения начала жечь лицо. кожу лица ст...,NaN,Positive,Negative,NaN,0,2,Power Nap SUPER MOISTURE EMBO HYDROGEL MASK,маски для лица,04/01/2025 00:25:15,приятно пахнуть,нанесение начало жечь лицо кожа лицо стянуть у...,
1,супер очищение,<NA>,очень хорошо очищает от загрязнений и черных т...,Neutral,NaN,Positive,1,5,Power Nap SUPER MOISTURE EMBO HYDROGEL MASK,маски для лица,04/01/2025 00:46:25,супер очищение,na,очень очищать загрязнение чёрный точка
2,"самая классная маска, беру несколько раз от да...",<NA>,буду заказывать еще!,Positive,NaN,Positive,1,5,Pury Fly SELF-BUBBLING COCONUT CHARCOAL MASK,маски для лица,04/01/2025 00:59:01,самый классный маска брать несколько дарлинг и...,na,быть заказывать
3,"Прекрасная маска, очень люблю ее использовать ...",<NA>,NaN,Positive,NaN,NaN,1,5,Magic Glow REVITALIZING MASK,маски для лица,04/01/2025 01:05:30,прекрасный маска очень любить использовать нан...,na,
4,Самые лучшие патчи которыми я когда-то пользов...,<NA>,NaN,Positive,NaN,NaN,1,5,LIFT ME UP,патчи для глаз,04/01/2025 01:09:16,самый хороший патч который пользоваться тонкий...,na,


## 3. Сохранение обработанного датасета

In [ ]:
df_out.to_csv("df_out.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("df_out.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Выделение прототипов лояльных и нелояльных отзывов

In [ ]:
best = df_out[
    ((df_out["pros_clean_sentiment"] == "Very Positive") |
    (df_out["pros_clean_sentiment"] == "Positive")) &
    (df_out["cons_clean_sentiment"].isna()) &
    ((df_out["comment_sentiment"] == "Very Positive") |
    (df_out["comment_sentiment"] == "Positive")) &
    (df_out["IsRecommended"] == 1) &
    (df_out["Stars"] == 5)
]
best.shape

(8494, 14)

In [ ]:
worst = df_out[(df_out["Stars"] < 4 ) & (df_out["IsRecommended"] == 0)]
worst.shape

(5450, 14)